# 二つの試行を記録し、保存して読み直す

Experimentを使うと、試した条件と得られたデータをRunごとにまとめて保存できます。このTutorialでは、二つの候補を評価し、その入力・結果・条件を一つのファイルに保存して、新しいPythonプロセスやNotebook kernelで読み直します。

Pythonと変数・目的関数・制約を知っていれば、このページだけで実行できます。[SDKのインストール](../getting_started/install.md)の手順1・2と、Notebookを実行する場合のkernel設定を済ませてください。使用するのは`ommx==3.0.0b7`で、Adapterやソルバー、レジストリの認証は不要です。

## 1. 小さな問題を用意する

二つの項目のうち、高々一つを選ぶ問題です。ここでは探索はせず、手元の二つの候補を評価します。`evaluate`は候補の目的値と実行可能性を計算し、`Solution`として返します。

In [1]:
from ommx import Instance
from ommx.experiment import Experiment

instance = Instance.maximize()
x = instance.new_binary("x")
y = instance.new_binary("y")
instance.objective = 3 * x + 2 * y
_ = instance.add_constraint(x + y <= 1)

## 2. 条件と結果をRunに記録する

Experimentは一連の試行をまとめる単位で、Runはその中の一回の試行です。共有する問題をExperimentに、候補の評価結果を各Runに添付します。`log_parameter`には後で一覧するスカラー値を明示的に記録します。

In [2]:
with Experiment() as experiment:
    experiment.log_instance("problem", instance)

    for label, candidate in [
        ("choose-x", {x.id: 1, y.id: 0}),
        ("choose-y", {x.id: 0, y.id: 1}),
    ]:
        with experiment.run() as run:
            solution = instance.evaluate(candidate)
            run.log_parameter("candidate", label)
            run.log_parameter("objective", solution.objective)
            run.log_parameter("feasible", solution.feasible)
            run.log_solution("evaluation", solution)

`with`を抜けるとRun、最後にExperimentの記録が確定します。記録はLocal RegistryというOMMXのローカル保存先に書かれます。ここでは保存先の構造や名前を操作する必要はありません。

## 3. 記録した条件を読む

In [3]:
experiment.run_parameters_df()

,candidate,feasible,objective
run_id,,,
0,choose-x,True,3.0
1,choose-y,True,2.0


両候補は実行可能で、`choose-x`の目的値は3、`choose-y`は2です。この表の列は`log_parameter`で記録した値です。メモリにある変数や実行環境が自動的にすべて記録されるわけではありません。

各Runに保存した`Solution`は、目的値だけでなく候補の変数値も持ちます。問題と評価結果のデータを残したので、表を表示しただけでなく、後で再び値を取り出せます。これは候補の評価であり、ソルバーが最適性を証明した記録ではありません。

## 4. ファイルに保存する

In [4]:
experiment.save("candidate-trials.ommx")

このファイルには、共有の問題、二つのRunの条件、各評価結果が入っています。以降はこのファイルだけを残して、作成に使ったPythonプロセスを終了してかまいません。同名のファイルは上書きされないので、再実行するときは保存先を別の名前にし、後の読込パスも合わせてください。

## 5. 新しいプロセスやkernelで開く

Notebookならkernelを再起動して、以下のセルから実行してください。スクリプトなら別のPythonプロセスで同じコードを実行します。カレントディレクトリに`candidate-trials.ommx`が必要です。ファイルを移動した場合は、そのパスに置き換えてください。

In [5]:
from ommx.experiment import Experiment

loaded = Experiment.import_archive("candidate-trials.ommx")
loaded.run_parameters_df()

,candidate,feasible,objective
run_id,,,
0,choose-x,True,3.0
1,choose-y,True,2.0


`import_archive`はファイルの内容を読み手側のLocal Registryに取り込みます。作成側の`experiment`や`instance`を引き継ぐ必要はありません。

In [6]:
problem = loaded.get_instance("problem")
for run in loaded.runs:
    saved = run.get_solution("evaluation")
    checked = problem.evaluate(saved.state)
    assert checked.objective == saved.objective
    assert checked.feasible == saved.feasible
    print(run.run_id, run.status, saved.objective, saved.feasible)

0 finished 3.0 True
1 finished 2.0 True


保存した二つのRunと入力・結果を読み直せました。`status`の`finished`はRunの処理が正常に終了したことを示し、最適性や実行可能性を表すものではありません。

ここまでで、試行の条件とデータを記録し、ファイルから取り出す流れは完了です。保存結果の読み直しと、同じ計算を同じ環境で再実行することは別です。後者に必要な入力、コード、設定、環境情報は、その目的に合わせて明示的に残します。

## 必要になったら

- [Experimentの記録とライフサイクル](../user_guide/experiment.md)：Runとソルバー呼び出しの関係、自動記録の範囲。
- [Experimentを継続・復帰する](../how_to/continue_experiment.md)：確定済みの記録からのforkとcheckpointからの復帰。
- [Experimentにデータを添付する](../how_to/attach_experiment_data.md)：JSON、ファイル、型付きデータ。
- [Artifactでデータを受け渡す](./share_in_ommx_artifact)：試行の階層を使わず、問題や結果を保存・交換する場合。